### Passo 1: Importações

In [2]:
from pyspark.sql import SparkSession
spark = SparkSession.builder.appName("Case").getOrCreate()

from pyspark.sql import functions as F
import pandas as pd

### Passo 2: Leitura das bases

In [3]:
PATH = '/home/tatiane/repositorios/data-projects/02_data_analysis/etl_banco_digital/'

agencias = spark.read.csv(f"{PATH}bronze/agencias.csv", header=True)
clientes = spark.read.parquet(f"{PATH}silver/clientes")
pix = spark.read.parquet(f"{PATH}silver/pix")
contas = spark.read.parquet(f"{PATH}silver/contas")
transacoes = spark.read.parquet(f"{PATH}silver/transacoes")

### Passo 3: Criação das bases analíticas

- Criação dos Dados cadastrais

In [4]:
# Criando variável idade e validando a transformação
df_idade = clientes.withColumn("idade", F.floor(F.months_between(F.current_date(), F.col("data_nascimento")) / 12))
df_idade.show(3)

[Stage 5:>                                                          (0 + 1) / 1]

+----------+----+---+-------+--------------+-------------+-------------------+-------------------------+-----+
|id_cliente|sexo| uf|  renda|        cidade|         nome|    data_nascimento|data_nascimento_formatada|idade|
+----------+----+---+-------+--------------+-------------+-------------------+-------------------------+-----+
|         1|   F| RJ|17706.0|       Niterói|Cliente_00001|1956-07-10 19:27:27|               10/07/1956|   70|
|        10|   M| PE|16978.0|     Petrolina|Cliente_00010|2001-01-21 11:44:14|               21/01/2001|   25|
|       100|   F| RJ|10797.0|Rio de Janeiro|Cliente_00100|1956-10-14 21:01:24|               14/10/1956|   69|
+----------+----+---+-------+--------------+-------------+-------------------+-------------------------+-----+
only showing top 3 rows


In [5]:
dados_cadastrais = df_idade.join(agencias, "uf", "left")
print('Qtd. registros:', dados_cadastrais.count(), '| Quantidade de colunas: ', len(dados_cadastrais.columns))
dados_cadastrais.printSchema()

Qtd. registros: 10000 | Quantidade de colunas:  11
root
 |-- uf: string (nullable = true)
 |-- id_cliente: string (nullable = true)
 |-- sexo: string (nullable = true)
 |-- renda: float (nullable = true)
 |-- cidade: string (nullable = true)
 |-- nome: string (nullable = true)
 |-- data_nascimento: string (nullable = true)
 |-- data_nascimento_formatada: string (nullable = true)
 |-- idade: long (nullable = true)
 |-- gerente_regional: string (nullable = true)
 |-- regiao: string (nullable = true)



- Criação da Conta

In [6]:
conta = contas.withColumnRenamed('status', 'status_conta')
conta = conta.drop('data_abertura')
conta = conta.withColumnRenamed('data_abertura_formatada', 'data_abertura')

- Criação do PIX

In [7]:
# 1. Realização da agregação por cliente (id_conta)
df_pix = pix.groupBy("id_conta").agg(
    F.count("id_pix").alias("quantidade_pix"),
    F.sum("valor").alias("valor_total_pix"),
    F.avg("valor").alias("ticket_medio_pix"),
    F.max("valor").alias("maior_pix"),
    F.min("valor").alias("menor_pix")
)

df_pix.show(3)

[Stage 10:>                                                         (0 + 2) / 2]

+--------+--------------+------------------+------------------+---------+---------+
|id_conta|quantidade_pix|   valor_total_pix|  ticket_medio_pix|maior_pix|menor_pix|
+--------+--------------+------------------+------------------+---------+---------+
|  100735|            13|13689.250007629395|1053.0192313561072|  3205.88|    52.67|
|  109180|            16|10921.730029582977| 682.6081268489361|  2041.78|     0.77|
|  104454|            19|17074.469924926758| 898.6563118382504|  3468.14|     74.8|
+--------+--------------+------------------+------------------+---------+---------+
only showing top 3 rows


In [ ]:
print('Qtd. registros:', clientes.count(), '| Quantidade de colunas: ', len(clientes.columns))
clientes.printSchema()

- Criação do Cartão

In [8]:
# 1. Realização da agregação por cliente (id_conta)
df_cartao = transacoes.groupBy("id_conta").agg(
    F.count("id_compra").alias("quantidade_compras"),
    F.sum("valor").alias("valor_total_cartao"),
    F.avg("valor").alias("ticket_medio_cartao"),
    F.max("valor").alias("maior_compra"),
    F.min("valor").alias("menor_compra")
)

df_cartao.show(3)

[Stage 13:=============================>                            (1 + 1) / 2]

+--------+------------------+------------------+-------------------+------------+------------+
|id_conta|quantidade_compras|valor_total_cartao|ticket_medio_cartao|maior_compra|menor_compra|
+--------+------------------+------------------+-------------------+------------+------------+
|  104603|                11|2306.9600105285645| 209.72363732077858|      470.54|       51.16|
|  108800|                 8| 1754.509994506836|  219.3137493133545|      389.46|      142.38|
|  104344|                 6| 987.4700164794922| 164.57833607991537|      294.07|       33.04|
+--------+------------------+------------------+-------------------+------------+------------+
only showing top 3 rows


### Junção das bases para criação de um único DataFrame

In [9]:
first_join = dados_cadastrais.join(conta, "id_cliente", "left")
print('Qtd. registros:', first_join.count(), '| Quantidade de colunas: ', len(first_join.columns))
first_join.printSchema()

Qtd. registros: 10000 | Quantidade de colunas:  15
root
 |-- id_cliente: string (nullable = true)
 |-- uf: string (nullable = true)
 |-- sexo: string (nullable = true)
 |-- renda: float (nullable = true)
 |-- cidade: string (nullable = true)
 |-- nome: string (nullable = true)
 |-- data_nascimento: string (nullable = true)
 |-- data_nascimento_formatada: string (nullable = true)
 |-- idade: long (nullable = true)
 |-- gerente_regional: string (nullable = true)
 |-- regiao: string (nullable = true)
 |-- id_conta: string (nullable = true)
 |-- tipo_conta: string (nullable = true)
 |-- status_conta: string (nullable = true)
 |-- data_abertura: string (nullable = true)



In [10]:
second_join = first_join.join(df_pix, "id_conta", "left")
print('Qtd. registros:', second_join.count(), '| Quantidade de colunas: ', len(second_join.columns))
second_join.printSchema()

Qtd. registros: 10000 | Quantidade de colunas:  20
root
 |-- id_conta: string (nullable = true)
 |-- id_cliente: string (nullable = true)
 |-- uf: string (nullable = true)
 |-- sexo: string (nullable = true)
 |-- renda: float (nullable = true)
 |-- cidade: string (nullable = true)
 |-- nome: string (nullable = true)
 |-- data_nascimento: string (nullable = true)
 |-- data_nascimento_formatada: string (nullable = true)
 |-- idade: long (nullable = true)
 |-- gerente_regional: string (nullable = true)
 |-- regiao: string (nullable = true)
 |-- tipo_conta: string (nullable = true)
 |-- status_conta: string (nullable = true)
 |-- data_abertura: string (nullable = true)
 |-- quantidade_pix: long (nullable = true)
 |-- valor_total_pix: double (nullable = true)
 |-- ticket_medio_pix: double (nullable = true)
 |-- maior_pix: float (nullable = true)
 |-- menor_pix: float (nullable = true)



In [11]:
final_join = second_join.join(df_cartao, "id_conta", "left")
print('Qtd. registros:', final_join.count(), '| Quantidade de colunas: ', len(final_join.columns))
final_join.printSchema()

Qtd. registros: 10000 | Quantidade de colunas:  25
root
 |-- id_conta: string (nullable = true)
 |-- id_cliente: string (nullable = true)
 |-- uf: string (nullable = true)
 |-- sexo: string (nullable = true)
 |-- renda: float (nullable = true)
 |-- cidade: string (nullable = true)
 |-- nome: string (nullable = true)
 |-- data_nascimento: string (nullable = true)
 |-- data_nascimento_formatada: string (nullable = true)
 |-- idade: long (nullable = true)
 |-- gerente_regional: string (nullable = true)
 |-- regiao: string (nullable = true)
 |-- tipo_conta: string (nullable = true)
 |-- status_conta: string (nullable = true)
 |-- data_abertura: string (nullable = true)
 |-- quantidade_pix: long (nullable = true)
 |-- valor_total_pix: double (nullable = true)
 |-- ticket_medio_pix: double (nullable = true)
 |-- maior_pix: float (nullable = true)
 |-- menor_pix: float (nullable = true)
 |-- quantidade_compras: long (nullable = true)
 |-- valor_total_cartao: double (nullable = true)
 |-- tick

#### Criação de novas colunas
- faixa de renda

In [12]:
df_create = final_join.withColumn(
    "faixa_renda",
    F.when(F.col("renda") <= 3000, "Baixa")
     .when((F.col("renda") > 3000) & (F.col("renda") <= 10000), "Media")
     .otherwise("Alta")
)
df_create.groupBy("faixa_renda").count().show()

+-----------+-----+
|faixa_renda|count|
+-----------+-----+
|       Alta| 6542|
|      Media| 2856|
|      Baixa|  602|
+-----------+-----+



- faixa etária

In [13]:
df_create = df_create.withColumn(
    "faixa_etaria",
    F.when(F.col("idade") <= 20, "Jovem")
     .when((F.col("idade") > 20) & (F.col("idade") <= 59), "Adulto")
     .otherwise("Idoso")
)
df_create.groupBy("faixa_etaria").count().show()

[Stage 38:>                                                         (0 + 1) / 1]

+------------+-----+
|faixa_etaria|count|
+------------+-----+
|       Jovem|  454|
|      Adulto| 7401|
|       Idoso| 2145|
+------------+-----+



- perfil financeiro

In [14]:
df_create = df_create.withColumn(
    "perfil_financeiro",
    F.when((F.col("valor_total_pix") + F.col("valor_total_cartao")) <= 3000, "Baixa")
     .when((F.col("valor_total_pix") + F.col("valor_total_cartao")) <= 10000, "Media")
     .otherwise("Alta")
)

df_create.groupBy("perfil_financeiro").count().show()

+-----------------+-----+
|perfil_financeiro|count|
+-----------------+-----+
|             Alta| 4590|
|            Media| 5367|
|            Baixa|   43|
+-----------------+-----+



- Cliente ativo

In [15]:
df_create.groupBy("status_conta").count().show()

+------------+-----+
|status_conta|count|
+------------+-----+
|   Bloqueada|  509|
|   Encerrada|  283|
|       Ativa| 9208|
+------------+-----+



In [16]:
df_create = df_create.withColumn(
    "cliente_ativo",
    F.when(F.col("status_conta") == "Ativa", "SIM").otherwise("Não"))
df_create.groupBy("cliente_ativo").count().show()

+-------------+-----+
|cliente_ativo|count|
+-------------+-----+
|          Não|  792|
|          SIM| 9208|
+-------------+-----+



In [21]:
len(df_create.columns)

29

In [18]:
df_create.printSchema()

root
 |-- id_conta: string (nullable = true)
 |-- id_cliente: string (nullable = true)
 |-- uf: string (nullable = true)
 |-- sexo: string (nullable = true)
 |-- renda: float (nullable = true)
 |-- cidade: string (nullable = true)
 |-- nome: string (nullable = true)
 |-- data_nascimento: string (nullable = true)
 |-- data_nascimento_formatada: string (nullable = true)
 |-- idade: long (nullable = true)
 |-- gerente_regional: string (nullable = true)
 |-- regiao: string (nullable = true)
 |-- tipo_conta: string (nullable = true)
 |-- status_conta: string (nullable = true)
 |-- data_abertura: string (nullable = true)
 |-- quantidade_pix: long (nullable = true)
 |-- valor_total_pix: double (nullable = true)
 |-- ticket_medio_pix: double (nullable = true)
 |-- maior_pix: float (nullable = true)
 |-- menor_pix: float (nullable = true)
 |-- quantidade_compras: long (nullable = true)
 |-- valor_total_cartao: double (nullable = true)
 |-- ticket_medio_cartao: double (nullable = true)
 |-- maio

### Selecionando as variaveis finais

In [24]:
df_final = df_create.select("id_cliente", "nome", "sexo", "idade", "uf", "cidade", "gerente_regional", "regiao", "renda", "tipo_conta", 
                           "status_conta", "data_abertura", "quantidade_pix", "valor_total_pix", "ticket_medio_pix", "maior_pix", 
                           "menor_pix", "quantidade_compras", "valor_total_cartao", "ticket_medio_cartao", "maior_compra", "menor_compra",
                          "faixa_renda", "perfil_financeiro", "faixa_etaria", "cliente_ativo")
df_final.printSchema()

root
 |-- id_cliente: string (nullable = true)
 |-- nome: string (nullable = true)
 |-- sexo: string (nullable = true)
 |-- idade: long (nullable = true)
 |-- uf: string (nullable = true)
 |-- cidade: string (nullable = true)
 |-- gerente_regional: string (nullable = true)
 |-- regiao: string (nullable = true)
 |-- renda: float (nullable = true)
 |-- tipo_conta: string (nullable = true)
 |-- status_conta: string (nullable = true)
 |-- data_abertura: string (nullable = true)
 |-- quantidade_pix: long (nullable = true)
 |-- valor_total_pix: double (nullable = true)
 |-- ticket_medio_pix: double (nullable = true)
 |-- maior_pix: float (nullable = true)
 |-- menor_pix: float (nullable = true)
 |-- quantidade_compras: long (nullable = true)
 |-- valor_total_cartao: double (nullable = true)
 |-- ticket_medio_cartao: double (nullable = true)
 |-- maior_compra: float (nullable = true)
 |-- menor_compra: float (nullable = true)
 |-- faixa_renda: string (nullable = false)
 |-- perfil_financeiro:

### Escrita

In [25]:
WRITE_PATH = '/home/tatiane/repositorios/data-projects/02_data_analysis/etl_banco_digital/gold'
df_final.write.parquet(f"{WRITE_PATH}/transacoes")